In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
#make sure you have the aggregated_data.csv
file_path = '/content/aggregated_data.csv'

# **Data Cleaning**
At the end of this it will generate the cleaned_data.csv automatically

In [24]:
# Cleaning and Validating ATL neighboorhood data
import pandas as pd

# Load the aggregated dataset from Google Drive
file_path = "/content/drive/MyDrive/Data Engineering Capstone/aggregated_data.csv"
df = pd.read_csv(file_path)

# Preview the data
print(df.head())

# Dataset Size
print("Dataset shape:", df.shape)

# Column Names
print("\nColumns:")
print(df.columns.tolist())

# Data Types
print("\nData types:")
print(df.dtypes)

# Missing Values

# Count missing values per column
missing = df.isnull().sum()

print("\nMissing values:")
print(missing)

# Remove OBJECTID only if every value is missing
if "OBJECTID" in df.columns:
    if df["OBJECTID"].isnull().all():
        df = df.drop(columns=["OBJECTID"])
        print("Removed empty OBJECTID column")

# Duplicate Rows
print("\nDuplicate rows:", df.duplicated().sum())

# Repeated Neighborhood Names
print("Duplicate names:", df["NAME"].duplicated().sum())

# Display Neighborhoods w/ repeated names
repeated = df[df["NAME"].duplicated(keep=False)]
print(repeated)

# Inconsistent Temperature Measurements
invalid_temp = df[
    (df["lst_min_c"] > df["lst_mean_c"]) |
    (df["lst_mean_c"] > df["lst_max_c"]) |
    (df["lst_std_c"] < 0)
]
print("\nInvalid temperature records:")
print(invalid_temp)

# Percentage Ranges
percentage_cols = [
    "tree_canopy_mean",
    "impervious_surfaces_mean",
    "pct_nlcd_water_11",
    "pct_nlcd_dev_open_21",
    "pct_nlcd_dev_low_22",
    "pct_nlcd_dev_med_23",
    "pct_nlcd_dev_high_24",
    "pct_nlcd_forest_total"
]

for col in percentage_cols:
    invalid = df[~df[col].between(0, 100)]
    print(f"{col}: {len(invalid)} invalid values")

# Check standard deviation values
std_cols = [
    "tree_canopy_std",
    "impervious_surfaces_std"
]

for col in std_cols:
    invalid = (df[col] < 0).sum()
    print(f"{col}: {invalid} negative values")

# Invalid neighborhood areas
invalid_area = df[df["area_sq_m"] <= 0]

print("\nInvalid neighborhood areas:")
print(invalid_area)

# Identify unusual average temperatures
Q1 = df["lst_mean_c"].quantile(0.25)
Q3 = df["lst_mean_c"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["lst_mean_c"] < lower) |
    (df["lst_mean_c"] > upper)
]

print("\nPotential temperature outliers:")
print(outliers[["NAME", "lst_mean_c"]])

# Confirm measurements passed validation
invalid_percentages = sum(
    (~df[col].between(0, 100)).sum()
    for col in percentage_cols
)

invalid_std = sum(
    (df[col] < 0).sum()
    for col in std_cols
)

if (
    len(invalid_temp) > 0
    or len(invalid_area) > 0
    or invalid_percentages > 0
    or invalid_std > 0
):
    raise ValueError(
        "Invalid measurements found. Review before exporting."
    )

# Save cleaned data without changing the original
df.to_csv("cleaned_data.csv", index=False)

print("\nCleaning complete!")
print("Final dataset shape:", df.shape)

# Summarize the final dataset
print("\nDATA CLEANING SUMMARY")

print("Total neighborhoods:", len(df))
print("Total columns:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Temperature issues:", len(invalid_temp))
print("Invalid areas:", len(invalid_area))
print("Potential temperature outliers:", len(outliers))

   OBJECTID                NAME NPU      area_sq_m  lst_mean_c  lst_min_c  \
0       NaN   Ridgewood Heights   C  554925.892427   37.765981  32.689996   
1       NaN            Fernleaf   C  220704.906603   41.167829  37.950006   
2       NaN  Westminster/Milmar   C  366360.227482   35.451942  28.709985   
3       NaN     Arden/Habersham   C  464192.480370   38.091447  35.909998   
4       NaN           Woodfield   C  187887.668886   36.021274  32.850000   

   lst_max_c  lst_std_c  tree_canopy_mean  tree_canopy_std  \
0  44.749994   2.094349         67.303571        22.320868   
1  46.129999   2.282861         71.402439        20.969505   
2  40.830011   3.134306         48.712531        26.643859   
3  43.110010   0.941459         59.540541        21.822584   
4  38.350000   1.402840         56.640777        23.674309   

   impervious_surfaces_mean  impervious_surfaces_std  pct_nlcd_water_11  \
0                 11.308442                12.523964           0.487013   
1             

# **Feature Engineering (Najja McGee)**

In [25]:
import pandas as pd

#direct path to your cleaned_data file inside the Capstone shortcut folder
file_path = '/content/drive/MyDrive/Data Engineering Capstone/cleaned_data.csv'

#read the dataset directly using pandas
df = pd.read_csv(file_path)

#view the first few rows of the data
df.head()

,NAME,NPU,area_sq_m,lst_mean_c,lst_min_c,lst_max_c,lst_std_c,tree_canopy_mean,tree_canopy_std,impervious_surfaces_mean,impervious_surfaces_std,pct_nlcd_water_11,pct_nlcd_dev_open_21,pct_nlcd_dev_low_22,pct_nlcd_dev_med_23,pct_nlcd_dev_high_24,pct_nlcd_forest_total
0,Ridgewood Heights,C,554925.892427,37.765981,32.689996,44.749994,2.094349,67.303571,22.320868,11.308442,12.523964,0.487013,55.519481,17.694805,2.110390,0.162338,24.025974
1,Fernleaf,C,220704.906603,41.167829,37.950006,46.129999,2.282861,71.402439,20.969505,13.479675,13.604967,0.000000,55.284553,23.170732,2.032520,0.000000,19.512195
2,Westminster/Milmar,C,366360.227482,35.451942,28.709985,40.830011,3.134306,48.712531,26.643859,24.314496,20.214801,0.000000,51.105651,35.872236,11.547912,1.228501,0.245700
3,Arden/Habersham,C,464192.480370,38.091447,35.909998,43.110010,0.941459,59.540541,21.822584,13.009653,9.426095,0.000000,78.378378,21.621622,0.000000,0.000000,0.000000
4,Woodfield,C,187887.668886,36.021274,32.850000,38.350000,1.402840,56.640777,23.674309,16.946602,17.432033,0.000000,59.223301,19.417476,6.796117,0.000000,14.563107


In [26]:
#recalculating tree canopy area and impervious surface area in square meters
#also divide the mean percentage by 100 to convert it to a decimal, then multiply by total area
df['tree_canopy_sq_m'] = (df['tree_canopy_mean'] / 100.0) * df['area_sq_m']
df['impervious_surfaces_sq_m'] = (df['impervious_surfaces_mean'] / 100.0) * df['area_sq_m']

#round the values to the nearest hundredth (2 decimal places)
df['tree_canopy_sq_m'] = df['tree_canopy_sq_m'].round(2)
df['impervious_surfaces_sq_m'] = df['impervious_surfaces_sq_m'].round(2)
df['area_sq_m'] = df['area_sq_m'].round(2)

#calculate tree canopy and impervious surface area per 1 square meter unit of neighborhood area
df['tree_canopy_per_sq_meter_unit'] = (df['tree_canopy_sq_m'] / df['area_sq_m']).round(4)
df['impervious_surfaces_per_sq_meter_unit'] = (df['impervious_surfaces_sq_m'] / df['area_sq_m']).round(4)


#drop the mean percentage columns and standard deviation
df_cleaned = df.drop(columns=['tree_canopy_mean', 'impervious_surfaces_mean', 'impervious_surfaces_std', 'tree_canopy_std'])


#show the first 5 rows the entire cleaned dataset
df_cleaned.head()

,NAME,NPU,area_sq_m,lst_mean_c,lst_min_c,lst_max_c,lst_std_c,pct_nlcd_water_11,pct_nlcd_dev_open_21,pct_nlcd_dev_low_22,pct_nlcd_dev_med_23,pct_nlcd_dev_high_24,pct_nlcd_forest_total,tree_canopy_sq_m,impervious_surfaces_sq_m,tree_canopy_per_sq_meter_unit,impervious_surfaces_per_sq_meter_unit
0,Ridgewood Heights,C,554925.89,37.765981,32.689996,44.749994,2.094349,0.487013,55.519481,17.694805,2.110390,0.162338,24.025974,373484.94,62753.47,0.6730,0.1131
1,Fernleaf,C,220704.91,41.167829,37.950006,46.129999,2.282861,0.000000,55.284553,23.170732,2.032520,0.000000,19.512195,157588.69,29750.30,0.7140,0.1348
2,Westminster/Milmar,C,366360.23,35.451942,28.709985,40.830011,3.134306,0.000000,51.105651,35.872236,11.547912,1.228501,0.245700,178463.34,89078.64,0.4871,0.2431
3,Arden/Habersham,C,464192.48,38.091447,35.909998,43.110010,0.941459,0.000000,78.378378,21.621622,0.000000,0.000000,0.000000,276382.71,60389.83,0.5954,0.1301
4,Woodfield,C,187887.67,36.021274,32.850000,38.350000,1.402840,0.000000,59.223301,19.417476,6.796117,0.000000,14.563107,106421.03,31840.58,0.5664,0.1695
